# Dry Bean Dataset — Exploratory Data Analysis

## 1. Introduction

### Project Overview

This project focuses on the exploratory data analysis (EDA) of the Dry Bean Dataset, which contains morphological measurements of different dry bean varieties.

The main objective is to investigate the structure, distributions, variability, and relationships between the dataset's features to better understand how different bean varieties can be distinguished.

### Objectives

- Explore the structure and characteristics of the dataset.
- Identify missing values and assess data quality.
- Calculate and interpret descriptive statistics.
- Visualize feature distributions and differences between bean varieties.
- Investigate correlations between numerical features.
- Identify potentially redundant features.
- Evaluate which features are most useful for distinguishing bean varieties.

## 2. Data Loading

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

dataPath = Path("../data/raw/Dry_Bean_Dataset.xlsx")
df = pd.read_excel(dataPath)

## Mini data research


In [2]:
df.head(5)
df.shape
numerical_features = df.select_dtypes(include="number")
numerical_features.shape
df.dtypes
df["Class"].unique()
print(df["Class"].value_counts(), "\n")
print(df["Class"].value_counts(normalize=True)* 100)
df.count() #non null values
df.isna().sum()

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64 

Class
DERMASON    26.052458
SIRA        19.366689
SEKER       14.892366
HOROZ       14.165014
CALI        11.975608
BARBUNYA     9.712732
BOMBAY       3.835133
Name: proportion, dtype: float64


Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

## Data selection, filtering and sorting

In [3]:
df.loc[:9,["Area", "Perimeter", "Class"]] #select rows and columns with labels
df.iloc[:9, [0,1,-1]] # select rows and columns by indexes
sekkerClass = pd.DataFrame(df[df["Class"] == "SEKER"])
sekkerClass["Class"].unique()
area = pd.DataFrame(df[df["Area"] > df["Area"].median()])
seker_above_global_median = df[(df["Class"] == "SEKER") & (df["Area"] > df["Area"].median())]
seker_above_global_median
df.sort_values("Area", ascending= False).head(10)
df.groupby(["Class"]).mean()["Area"]


Class
BARBUNYA     69804.133132
BOMBAY      173485.059387
CALI         75538.211043
DERMASON     32118.710942
HOROZ        53648.508817
SEKER        39881.299951
SIRA         44729.128604
Name: Area, dtype: float64

## 5. Grouping and Aggregation

In [4]:
meanAndMed = pd.DataFrame(df.groupby(["Class"]).mean()["Area"])
meanAndMed["Median"] = df.groupby(["Class"]).median()["Area"]
meanAndMed

minMaxStd = pd.DataFrame(df.groupby(["Class"]).min()["Perimeter"])
minMaxStd["Max"] = df.groupby(["Class"]).max()["Perimeter"]
minMaxStd["Std"] = df.groupby(["Class"]).std()["Perimeter"]
minMaxStd.rename(columns={"Area":"Min"},inplace=True)
minMaxStd

areaRatios = pd.DataFrame(df.groupby(["Class"])["Area"].agg(["mean","median","std","min", "max"])).reset_index()
areaRatios.sort_values("mean", ascending=False)

areaRatios.sort_values("std", ascending=False).head(1)



C:\Users\piotr\AppData\Local\Temp\ipykernel_20792\3508442050.py:8: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  minMaxStd.rename(columns={"Area":"Min"},inplace=True)


,Class,mean,median,std,min,max
1,BOMBAY,173485.059387,171494.5,23327.688116,114004,254616


## 6. Descriptive Statistics

In [5]:
mean = df["Area"].mean()
median = df["Area"].median()
variance = df["Area"].var()
std = df["Area"].std()

q1, q3 = df["Area"].quantile([0.25, 0.75])
iqr = q3 - q1

cv = pd.DataFrame(df.groupby(["Class"])["Area"].agg(["std","mean"])).reset_index()
cv["cv"] = cv["std"] / cv["mean"] * 100
cv = cv.drop(columns=["std", "mean"])
cv

area_stats = df.groupby("Class")["Area"].agg(mean="mean", median="median", std="std", iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)).reset_index()

ranking_df = df.groupby("Class")["Area"].agg(["std", "mean"]).reset_index()
ranking_df["CV"] = ranking_df["std"] / ranking_df["mean"] * 100
ranking_df = ranking_df.drop(columns=["mean"])
ranking_df["rankStd"] = ranking_df["std"].rank(ascending=False, method='min')
ranking_df["rankCV"] = ranking_df["CV"].rank(ascending=False, method='min')
ranking_df.sort_values("rankStd", inplace=True)
ranking_df

arr = df["Area"].to_numpy()
np.mean(arr)
np.std(arr, ddof=1)
np.var(arr, ddof=1)
np.quantile(arr, .25)
np.quantile(arr, .75)

C:\Users\piotr\AppData\Local\Temp\ipykernel_20792\2812379637.py:21: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  ranking_df.sort_values("rankStd", inplace=True)


np.float64(61332.0)

### 7. NumPy Array Operations

In [6]:
array = df.to_numpy()
array.shape
array.ndim # number of dimension
array.size # no of fields
array.dtype # object

areaIndx = df.columns.get_loc("Area")
areaArray = array[:,areaIndx]
areaAvg = np.mean(areaArray)
overAvgAreaMask = areaArray > areaAvg
array[overAvgAreaMask]

standardArea = (areaArray - areaAvg)/np.std(areaArray)
np.mean(standardArea)
np.std(standardArea)

new_col = np.where(areaArray > np.median(areaArray), 1, 0)
areaOverMedian = np.column_stack((areaArray, new_col))
print(np.median(areaArray))
areaOverMedian

area20 = areaArray[:20]
matrix45 = area20.reshape(4,5)
matrix45

44652.0


array([[28395, 28734, 29380, 30008, 30140],
       [30279, 30477, 30519, 30685, 30834],
       [30917, 31091, 31107, 31158, 31158],
       [31178, 31202, 31203, 31272, 31335]], dtype=object)